# Extra credit `.cell` 특징 추출 — Google Drive에서 자동 실행

공유 [extra credit 데이터 폴더](https://drive.google.com/drive/folders/1eLe4olc2sjcBnyaOKdMfKzAi7-RQhLoW)의 ZIP을 파일명으로 찾아 Colab 임시 저장소에 다운로드한 뒤, 수정한 `preprocess_extra_credit.py`로 특징을 추출하고 **내 드라이브**에 결과를 저장합니다. 로컬 PC에서 ZIP을 올릴 필요가 없습니다.

**권장:** 먼저 `SCALE = "pilot"`으로 작은 조합을 확인하고, 최종 결과는 `SCALE = "full"`로 실행하세요. 전체와 축소 ZIP을 합쳐 사용하지 않습니다. 두 실행은 사이트·기록 수가 다르므로 점수 비교 시 규모를 표시하세요. `trim0`만 실행합니다. Colab에서 위에서 아래로 순서대로 실행하고 Google 계정 접근 승인을 진행하세요.


In [ ]:
# 설정: 사전 점검은 "pilot", 95개 사이트 전체 실험은 "full"
SCALE = "full"
SOURCE_FOLDER_ID = "1eLe4olc2sjcBnyaOKdMfKzAi7-RQhLoW"
OUTPUT_ROOT = "/content/drive/MyDrive/MLB-Project/feature/extra_credit"

ZIP_NAMES = {"pilot": ("mon_25", "unmon_5000.zip"),
             "full": ("mon.zip", "unmon.zip")}
assert SCALE in ZIP_NAMES
print("선택한 조합:", SCALE, ZIP_NAMES[SCALE])


In [ ]:
from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive')

from google.auth import default
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from pathlib import Path
import shutil, subprocess, sys, zipfile

credentials, _ = default()
service = build('drive', 'v3', credentials=credentials, cache_discovery=False)
result = service.files().list(
    q=f"'{SOURCE_FOLDER_ID}' in parents and trashed = false",
    fields='nextPageToken,files(id,name,size,mimeType)', pageSize=1000,
    supportsAllDrives=True, includeItemsFromAllDrives=True
).execute()
if result.get('nextPageToken'):
    raise RuntimeError('폴더 목록이 1000개를 넘어 일부만 검색했습니다.')
files = result.get('files', [])
print('접근 가능한 폴더 항목:', [f['name'] for f in files])
selected = {}
for expected in ZIP_NAMES[SCALE]:
    matches = [f for f in files if f['name'] == expected]
    if len(matches) != 1:
        raise RuntimeError(f'{expected!r} 파일을 정확히 하나 찾지 못했습니다. 같은 Google 계정으로 공유 폴더를 열 수 있는지 확인하세요.')
    selected[expected] = matches[0]

work = Path('/content/extra_credit_work')
work.mkdir(parents=True, exist_ok=True)
paths = {}
for expected, meta in selected.items():
    # Drive의 mon_25는 확장자가 없지만 내용은 ZIP입니다.
    target = work / (expected if expected.endswith('.zip') else expected + '.zip')
    with target.open('wb') as out:
        downloader = MediaIoBaseDownload(out, service.files().get_media(fileId=meta['id']), chunksize=8*1024*1024)
        done = False
        while not done:
            progress, done = downloader.next_chunk()
            print(f"{expected}: {progress.progress()*100:.1f}%", end='\r')
    print(f"{expected}: 다운로드 완료 ({target.stat().st_size:,} bytes)")
    if int(meta['size']) != target.stat().st_size or not zipfile.is_zipfile(target):
        raise RuntimeError(f'{expected}: 크기가 다르거나 ZIP 형식이 아닙니다.')
    paths[expected] = target


In [ ]:
EXTRACTOR_SOURCE = '#!/usr/bin/env python3\n"""Extra-credit .cell -> main-experiment-compatible 110-feature tables.\n\nBased on team \'공통 규칙 문서 정의\' (2026-11-23), pp. 2-8.\nKeep split and join samples in separate outputs. Empty samples have zero-valued\nfeatures and are explicitly marked in both the feature table and audit log.\n"""\nimport argparse, csv, gzip, io, json, re, zipfile\nfrom collections import Counter\nfrom pathlib import Path\nimport numpy as np\n\nSTAT5=[\'mean\',\'std\',\'max\',\'min\',\'median\']\nSTAT6=[\'mean\',\'std\',\'max\',\'q25\',\'q50\',\'q75\']\nFEATURES=([\'g1_n_total\',\'g1_n_out\',\'g1_in_ratio\']\n          +[\'g2_out_pos_mean\',\'g2_out_pos_std\']\n          +[f\'g2_chunk{n}_{s}\' for n in (20,50,100) for s in STAT5]\n          +[\'g3_n_out_bursts\',\'g3_bursts_len1_5\',\'g3_bursts_len6_20\',\n            \'g3_bursts_len21p\',\'g3_max_out_burst\',\'g3_max_in_burst\']\n          +[\'g4_first30_out_ratio\',\'g4_first100_out_ratio\',\'g4_last30_out_ratio\']\n          +[\'g5_duration\']+[f\'g5_pps_{s}\' for s in STAT5]\n          +[\'g5_rel_t25\',\'g5_rel_t50\',\'g5_rel_t75\']\n          +[f\'g5_iat_all_{s}\' for s in STAT6]\n          +[f\'g5_iat_out_{s}\' for s in STAT6]\n          +[f\'g5_iat_in_{s}\' for s in STAT6]\n          +[\'g5_iat_zero_ratio\']\n          +[f\'g5_iat_pos_{s}\' for s in (\'mean\',\'std\',\'max\')]\n          +[f\'g6_cumul_{i:02}\' for i in range(50)])\nMETA=[\'record_id\',\'source_group_id\',\'variant\',\'status\',\'dataset\',\'page_id\',\'sample_idx\',\'label\',\'trim\']\nassert len(FEATURES)==110, len(FEATURES)\nPAT_MON=re.compile(r\'^(\\d+)-(\\d+)_(split_[0-4]|join)\\.cell$\')\nPAT_UNMON=re.compile(r\'^(\\d+)_(split_[0-4]|join)\\.cell$\')\n\ndef stats(x, six=False):\n    if len(x)==0: return [0.0]*(6 if six else 5)\n    x=np.asarray(x,dtype=float)\n    if six: return [float(np.mean(x)),float(np.std(x)),float(np.max(x)),float(np.quantile(x,.25)),float(np.median(x)),float(np.quantile(x,.75))]\n    return [float(np.mean(x)),float(np.std(x)),float(np.max(x)),float(np.min(x)),float(np.median(x))]\n\ndef parse_cell(data):\n    if not data.strip(): return np.empty((0,2),dtype=float),\'empty\'\n    # Only two necessary columns: time and direction; third (signed size) checked for format.\n    try:\n        a=np.loadtxt(io.BytesIO(data),dtype=float,ndmin=2)\n        if a.shape[1]!=3: return np.empty((0,2),dtype=float),\'bad_column_count\'\n        if not np.isfinite(a).all(): return np.empty((0,2),dtype=float),\'nonfinite\'\n        if not np.isin(a[:,1],(-1,1)).all(): return np.empty((0,2),dtype=float),\'bad_direction\'\n        if np.any(np.diff(a[:,0]) < -1e-8): return np.empty((0,2),dtype=float),\'nonmonotonic_time\'\n        return a[:,:2],\'ok\'\n    except Exception: return np.empty((0,2),dtype=float),\'parse_error\'\n\ndef chunks_out(d,n):\n    # Match feature_extraction.ipynb: a short record forms one complete fallback chunk.\n    k=len(d)//n\n    if k==0:\n        counts=np.asarray([np.count_nonzero(d==1)],dtype=float)\n    else:\n        counts=np.asarray([np.count_nonzero(d[i:i+n]==1) for i in range(0,k*n,n)],dtype=float)\n    return stats(counts)\n\ndef feats(arr,trim):\n    a=arr[trim:]\n    n=len(a)\n    if not n: return [0.0]*110\n    t=a[:,0]-a[0,0]\n    d=a[:,1]\n    out=(d==1); incoming=(d==-1)\n    vals=[float(n),float(np.sum(out)),float(np.sum(incoming)/n)]\n    # Outgoing positions counted from 0. Team document doesn\'t resolve normalized/raw index.\n    positions=np.nonzero(out)[0].astype(float)\n    vals += [float(np.mean(positions)) if len(positions) else 0., float(np.std(positions)) if len(positions) else 0.]\n    for chunk in (20,50,100): vals+=chunks_out(d,chunk)\n    # Consecutive same-direction runs. Count outbound runs and runs of either direction.\n    edges=np.r_[0,np.where(np.diff(d)!=0)[0]+1,n]\n    lens=np.diff(edges); run_dirs=d[edges[:-1]]\n    vals += [float(np.sum(run_dirs==1)),float(np.sum((lens>=1)&(lens<=5))),float(np.sum((lens>=6)&(lens<=20))),\n             float(np.sum(lens>20)),float(np.max(lens[run_dirs==1])) if np.any(run_dirs==1) else 0.,\n             float(np.max(lens[run_dirs==-1])) if np.any(run_dirs==-1) else 0.]\n    vals += [float(np.mean(out[:k])) for k in (30,100)]\n    vals += [float(np.mean(out[-30:]))]\n    duration=float(t[-1])\n    vals += [duration]\n    pps=np.bincount(np.floor(np.maximum(t,0)).astype(int))\n    vals += stats(pps)\n    vals += [float(t[min(n-1,max(0,int(np.ceil(n*q))-1))]/duration) if duration>0 else 0. for q in (.25,.5,.75)]\n    all_iat=np.diff(t); out_iat=np.diff(t[out]); in_iat=np.diff(t[incoming]); positive=all_iat[all_iat>0]\n    for x in (all_iat,out_iat,in_iat): vals+=stats(x,True)\n    vals += [float(np.mean(all_iat==0)) if len(all_iat) else 0.]\n    vals += [float(np.mean(positive)) if len(positive) else 0.,\n             float(np.std(positive)) if len(positive) else 0.,\n             float(np.max(positive)) if len(positive) else 0.]\n    cumul=np.cumsum(d)\n    vals += np.interp(np.linspace(0,n-1,50),np.arange(n),cumul).tolist()\n    assert len(vals)==110, len(vals)\n    return vals\n\ndef run(inputs,outdir,trims):\n    outdir.mkdir(parents=True,exist_ok=True)\n    if len(set(trims))!=len(trims) or any(k<0 for k in trims):\n        raise ValueError(\'Trims must be distinct nonnegative integers\')\n    outputs={(variant,k):gzip.open(outdir/f\'features_extra_{variant}_trim{k}.csv.gz\',\'wt\',newline=\'\',encoding=\'utf-8\',compresslevel=5)\n             for variant in (\'split\',\'join\') for k in trims}\n    writers={key:csv.writer(f) for key,f in outputs.items()}\n    for w in writers.values(): w.writerow(META+FEATURES)\n    audit=open(outdir/\'audit.csv\',\'w\',newline=\'\',encoding=\'utf-8\')\n    aw=csv.writer(audit); aw.writerow([\'record_id\',\'source_group_id\',\'dataset\',\'variant\',\'source_file\',\'raw_packets\',\'status\'])\n    counts=Counter(); variant_counts=Counter(); group_seen=set(); empty_after_trim=Counter()\n    try:\n        for kind,path in inputs:\n            pattern=PAT_MON if kind==\'mon\' else PAT_UNMON\n            with zipfile.ZipFile(path) as z:\n                for member in z.infolist():\n                    if member.is_dir() or not member.filename.endswith(\'.cell\'): continue\n                    name=Path(member.filename).name\n                    m=pattern.fullmatch(name)\n                    if not m:\n                        counts[\'filename_not_recognized\']+=1; continue\n                    if kind==\'mon\': page,sample,variant=int(m[1]),int(m[2]),m[3]; label=page;group=f\'mon_{page}-{sample}\'\n                    else: page,sample,variant=-1,int(m[1]),m[2];label=-1;group=f\'unmon_{sample}\'\n                    rid=f\'{group}_{variant}\'\n                    arr,status=parse_cell(z.read(member))\n                    if status not in (\'ok\',\'empty\'):\n                        raise ValueError(f\'{member.filename}: {status}; refusing to write zero features for invalid data\')\n                    aw.writerow([rid,group,kind,variant,member.filename,len(arr),status])\n                    counts[f\'{kind}_{status}\']+=1\n                    variant_counts[(kind,variant)]+=1\n                    group_seen.add(group)\n                    for k in trims:\n                        trimmed_empty=len(arr)<=k\n                        if trimmed_empty: empty_after_trim[(kind,variant,k)]+=1\n                        row=[rid,group,variant,\'empty_after_trim\' if trimmed_empty and status==\'ok\' else status,\n                             kind,page,sample,label,k]+feats(arr,k)\n                        writers[(\'join\' if variant==\'join\' else \'split\',k)].writerow(row)\n                    if counts[f\'{kind}_{status}\']%5000==0 and status==\'ok\': print(f\'{kind}: {sum(v for key,v in counts.items() if key.startswith(kind+"_"))} files\',flush=True)\n    finally:\n        for f in outputs.values(): f.close()\n        audit.close()\n    report={\'zip_inputs\':{k:str(v.name) for k,v in inputs},\'trims\':trims,\'feature_count\':len(FEATURES),\'metadata_columns\':META,\n            \'row_count_per_trim\':sum(variant_counts.values()),\n            \'source_group_count\':len(group_seen),\'status_counts\':dict(counts),\'variant_counts\':{f\'{k[0]}_{k[1]}\':v for k,v in sorted(variant_counts.items())},\n            \'empty_after_trim\':{f\'{kind}_{variant}_trim{k}\':v for (kind,variant,k),v in sorted(empty_after_trim.items())},\n            \'notes\':[\'Same 110 feature names and formulas as feature_extraction.ipynb; compare source datasets before cross-dataset benchmarking.\',\n                    \'Empty samples have zero-valued features and explicit status; decide whether to retain them for modeling.\',\n                    \'Split train/test by source_group_id, not record_id; source_group_id is in each feature CSV.\',\n                    \'Join and split samples are saved in separate feature CSVs.\',\n                    \'Do not combine nested mon subsets or nested unmon subsets without de-duplication.\']}\n    (outdir/\'validation.json\').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n    print(json.dumps(report,ensure_ascii=False,indent=2))\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--mon\',type=Path,required=True);p.add_argument(\'--unmon\',type=Path,required=True);p.add_argument(\'--out\',type=Path,default=Path(\'out\'));p.add_argument(\'--trims\',type=int,nargs=\'+\',default=[0,10,20]);a=p.parse_args()\n    run([(\'mon\',a.mon),(\'unmon\',a.unmon)],a.out,a.trims)\n'


In [ ]:
# 검증된 전처리 코드의 고정 사본을 Colab에 기록합니다.
from pathlib import Path
extractor = Path('/content/extra_credit_work/preprocess_extra_credit.py')
extractor.write_text(EXTRACTOR_SOURCE, encoding='utf-8')
output = Path('/content/extra_credit_work/results')
mon_name, unmon_name = ZIP_NAMES[SCALE]
subprocess.run([sys.executable, str(extractor), '--mon', str(paths[mon_name]),
                '--unmon', str(paths[unmon_name]), '--out', str(output), '--trims', '0'], check=True)


In [ ]:
# 성공한 결과만 내 Google Drive에 저장합니다. 기존 결과 폴더는 덮어쓰지 않습니다.
import csv, gzip, json
report = json.loads((output/'validation.json').read_text(encoding='utf-8'))
expected_groups = 10000 if SCALE == 'pilot' else 38000
assert report['source_group_count'] == expected_groups, report['source_group_count']
assert report['row_count_per_trim'] == expected_groups * 6
assert report['feature_count'] == 110
assert set(report['status_counts']) <= {'mon_ok','mon_empty','unmon_ok','unmon_empty'}
for kind, n in [('split',expected_groups*5),('join',expected_groups)]:
    with gzip.open(output/f'features_extra_{kind}_trim0.csv.gz','rt',encoding='utf-8') as f:
        header = next(csv.reader(f))
    assert len(header) == 119 and 'source_group_id' in header and 'variant' in header
    assert sum(report['variant_counts'][f'{ds}_{v}'] for ds in ('mon','unmon')
               for v in (('join',) if kind == 'join' else tuple(f'split_{i}' for i in range(5)))) == n

destination = Path(OUTPUT_ROOT)/f'{SCALE}_trim0'
if destination.exists():
    raise FileExistsError(f'{destination} 폴더가 이미 있습니다. 기존 결과 보호를 위해 저장을 중단했습니다. 폴더명을 바꾸거나 기존 결과를 확인하세요.')
destination.mkdir(parents=True)
for name in ['features_extra_split_trim0.csv.gz','features_extra_join_trim0.csv.gz','audit.csv','validation.json']:
    shutil.copy2(output/name, destination/name)
print('Google Drive 저장 완료:', destination)
print('분할 실험 입력:', destination/'features_extra_split_trim0.csv.gz')
print('원본 방문 그룹:', report['source_group_count'], '행/trim:', report['row_count_per_trim'])
print('빈 원본 파일:', {k:v for k,v in report['status_counts'].items() if k.endswith('_empty')})
